In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import os
import time
from google.colab import drive

# --- 1. Mount Google Drive ---
print("--- Step 1: Mounting Google Drive ---")
drive.mount('/content/drive')


# --- 2. Define Paths ---
# SLOW path on your Google Drive
DRIVE_SOURCE_DIR = '/content/drive/MyDrive/data/data'

# FAST path on the local Colab disk
LOCAL_DATA_DIR = '/content/dataset'

# Where to save the final, trained model
MODEL_SAVE_PATH = '/content/drive/MyDrive/data/data/chest_xray_detector_mobilenet.keras'


# --- 3. Copy Data from Drive to Local Disk (The Speed Fix) ---
print(f"\n--- Step 2: Copying dataset from {DRIVE_SOURCE_DIR} ---")
print("This may take a few minutes, but you only do it once.")
start_time = time.time()

# Create the local destination folder
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Use the 'cp' command to copy. It's much faster than Python's 'shutil'
# -r = recursive (all subfolders)
# -n = no-clobber (don't re-copy files that already exist)
!cp -r -n "{DRIVE_SOURCE_DIR}"/* "{LOCAL_DATA_DIR}"/

print(f"Copying complete! Took {time.time() - start_time:.2f} seconds.")


# --- 4. Configure GPU ---
print("\n--- Step 3: Configuring GPU ---")
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    print(f"Found {len(gpus)} GPUs.")
    try:
        # Enable 'memory growth'
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled for GPUs.")
    except RuntimeError as e:
        print(e)
else:
    print("WARNING: No GPU found. Training will use the CPU.")


# --- 5. Set Model Configuration ---
print("\n--- Step 4: Setting Configuration ---")
IMG_WIDTH = 128
IMG_HEIGHT = 128
BATCH_SIZE = 256  # <-- INCREASED BATCH SIZE to feed the GPU
EPOCHS = 10
print(f"Batch size set to: {BATCH_SIZE}")


# --- 6. Load Data (from the FAST local path) ---
print(f"\n--- Step 5: Loading data from {LOCAL_DATA_DIR} ---")
train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_DATA_DIR,  # <-- USING FAST LOCAL PATH
    validation_split=0.2,
    subset="both",
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

class_names = train_ds.class_names
print(f"Classes found: {class_names}")


# --- 7. Preprocess Data ---
print("\n--- Step 6: Preprocessing data for MobileNetV2 ---")
def preprocess_data(image, label):
    return preprocess_input(image), label

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(preprocess_data, num_parallel_calls=AUTOTUNE).cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.map(preprocess_data, num_parallel_calls=AUTOTUNE).cache().prefetch(buffer_size=AUTOTUNE)
print("Dataset preprocessing and caching complete.")


# --- 8. Build the Model with Transfer Learning ---
print("\n--- Step 7: Building MobileNetV2 model ---")
base_model = MobileNetV2(
    input_shape=(IMG_WIDTH, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze the base model
base_model.trainable = False

# Add our custom head
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])


# --- 9. Compile the Model ---
print("\n--- Step 8: Compiling model ---")
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()


# --- 10. Train the Model ---
print("\n--- Step 9: Starting model training (This should be fast now) ---")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


# --- 11. Save the Final Model to Google Drive ---
print(f"\n--- Step 10: Saving trained model to {MODEL_SAVE_PATH} ---")
model.save(MODEL_SAVE_PATH)
print(f"Model training complete! Saved to {MODEL_SAVE_PATH}")

--- Step 1: Mounting Google Drive ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Step 2: Copying dataset from /content/drive/MyDrive/data/data ---
This may take a few minutes, but you only do it once.
Copying complete! Took 303.50 seconds.

--- Step 3: Configuring GPU ---
Found 1 GPUs.
Memory growth enabled for GPUs.

--- Step 4: Setting Configuration ---
Batch size set to: 256

--- Step 5: Loading data from /content/dataset ---
Found 10838 files belonging to 2 classes.
Using 8671 files for training.
Using 2167 files for validation.
Classes found: ['chest_xray', 'not_xray']

--- Step 6: Preprocessing data for MobileNetV2 ---
Dataset preprocessing and caching complete.

--- Step 7: Building MobileNetV2 model ---
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

--- Step 8: Compiling model ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


--- Step 9: Starting model training (This should be fast now) ---
Epoch 1/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 91s 2s/step - accuracy: 0.8067 - loss: 0.4029 - val_accuracy: 0.9995 - val_loss: 0.0148
Epoch 2/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.9996 - loss: 0.0143 - val_accuracy: 0.9995 - val_loss: 0.0072
Epoch 3/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.9996 - loss: 0.0083 - val_accuracy: 0.9995 - val_loss: 0.0052
Epoch 4/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.9995 - loss: 0.0062 - val_accuracy: 0.9995 - val_loss: 0.0040
Epoch 5/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.9997 - loss: 0.0046 - val_accuracy: 0.9995 - val_loss: 0.0031
Epoch 6/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.9997 - loss: 0.0038 - val_accuracy: 0.9995 - val_loss: 0.0026
Epoch 7/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.9996 - loss: 0.0032 - val_accuracy: 1.0000 - val_loss: 0.0021
Epoch 8/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms